# Rust 116: Concurrency and Async/Await — Practice

Companion exercises for `16_concurrency-and-async_e.ipynb`. Every section mirrors the section of the same number in the example notebook, and every demonstration cell has an exercise, so a `**See**` reference lands on the exact code that shows the technique.

The checks at the bottom of each cell are the grader. Edit only the `todo!()` bodies; when a cell runs without panicking you will see an `Exercise N.M passed` line.


### Contents

- [0. How to Complete These Exercises](#0-how-to-complete-these-exercises)
- [1. OS Threads: `spawn`, `join`, `Builder`](#1-os-threads-spawn-join-builder)
- [2. `Send` and `Sync`: The Compile-Time Race Proof](#2-send-and-sync-the-compile-time-race-proof)
- [3. Channels: Ownership as the Message Protocol](#3-channels-ownership-as-the-message-protocol)
- [4. `Mutex` and `RwLock`: Shared State Under Guard](#4-mutex-and-rwlock-shared-state-under-guard)
- [5. Atomics: Lock-Free Counters and Flags](#5-atomics-lock-free-counters-and-flags)
- [7. `async`/`await`: Futures Are Frames](#7-asyncawait-futures-are-frames)
- [8. The Runtime: `tokio`, `spawn`, `JoinHandle`](#8-the-runtime-tokio-spawn-joinhandle)
- [9. `select!` and `timeout`: Racing Futures](#9-select-and-timeout-racing-futures)
- [10. Blocking vs Async: The Rules of the Road](#10-blocking-vs-async-the-rules-of-the-road)
- [11. Mixed Review](#11-mixed-review)

§1–§10 each have exercises for the demonstration cells of the example notebook; §6 is reading-only (the §6.1 timing cell is the demonstration, and this notebook never asserts a timing); §12 says where the track goes next.


## 0. How to Complete These Exercises

This notebook runs on the same **evcxr** kernel as the example notebook. Three consequences shape how you work:

1. **No `fn main`.** The body of a cell *is* the body of `main`. Type statements directly.
2. **State persists.** A binding from an earlier cell is visible in every cell below it — including `use` lines.
3. **Items are order-independent.** A struct or fn may be used above the cell that defines it.

> **Item names must be unique across the whole notebook.** Every cell shares one namespace; two functions with the same name are a redefinition error even in different sections. The exercise names here are unique by construction — do not introduce colliding helpers. The same rule covers `use` lines: the first cell that needs a name is the one that imports; later cells rely on persistence — a repeated `use` is E0252.

### The stub convention

Each exercise cell declares signatures with contract doc comments and a `// --- check (do not modify) ---` block of asserts. The body you must fill in is `todo!()`, which compiles but **panics** when executed. Replace it and re-run; an `Exercise N.M passed` line means you are done.

### Four notes specific to this chapter

- **Runtime per cell, the manual form.** Every async exercise builds `tokio::runtime::Runtime::new()` and `block_on`s what it needs, then lets the runtime drop — the manual form of `#[tokio::main]`, because the kernel owns `main` (§0 of the example notebook). A sync stub that needs async builds its own runtime; an async stub is driven by the check's `block_on`.
- **Determinism without timing asserts.** Threads are joined before any assert; channel results are collected to completion; the only sleeps have 10×+ margins (5ms against 50ms). No exercise asserts a *duration* — §6's thread/task ratio is the example notebook's lesson, not a check here.
- **Imports arrive once, with the first cell that needs them.** `thread` lands in §1.1, `Arc` in §2.1, the atomic types in §5.1, `Duration` in §9.2 — every later cell relies on persistence, exactly as the example notebook teaches in its §0.
- **The crate cell below is given and must run first** (once per kernel session, features included). A `cannot find crate` diagnostic means it has not run — restart and go from the top.

### Expect warnings until you finish

Three warning families are normal and self-cleaning:

- `unused variable` — every unsolved stub whose parameters are untouched;
- `unused import` — the `use` lines serving a stub you have not solved yet;
- **stub-induced dead code** (`never used` / `never constructed`) — given helpers that only a *solution* calls, like §7's `add`/`hold_string`/`identity_unit`.

Chapter 14's fourth family (`variable does not need to be mutable`) is designed out: no stub takes a `mut` binding parameter. Anything outside these families in your solve is a real smell.


In [ ]:
// 0.1 The chapter's crate cell: run once per kernel session, features included.
:dep tokio = { version = "1", features = ["full"] }

let rt = tokio::runtime::Runtime::new().unwrap();
let answer = rt.block_on(async { 40 + 2 });
assert_eq!(answer, 42);
println!("tokio loaded; block_on smoke: {}", answer);


## 1. OS Threads: `spawn`, `join`, `Builder`

A thread is a function call that happens elsewhere; `join()` is how you get the answer back and find out whether it panicked.

**See**: §1 of the example notebook.


In [ ]:
// Given: the thread module persists across cells once imported here.
use std::thread;

/// Double a fill quantity. The trivial work every thread below will do.
fn double(x: u64) -> u64 {
    todo!()
}

/// Run `double` on ANOTHER thread and bring the answer back through the
/// JoinHandle: spawn (move the argument in), join, unwrap the Result.
fn run_on_thread(x: u64) -> u64 {
    todo!()
}

/// How much hardware does the OS offer? available_parallelism returns a
/// NonZeroUsize inside a Result; map it to a plain usize (0 on error).
fn count_cores() -> usize {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(double(21), 42);
assert_eq!(run_on_thread(21), 42); // the same 42, computed elsewhere
assert!(count_cores() > 0);
println!("Exercise 1.1 passed");


## 2. `Send` and `Sync`: The Compile-Time Race Proof

`Arc` crosses threads; `Rc` is an E0277 before the program exists. `thread::scope` extends that guarantee to *borrows* of data you still own.

**See**: §2 of the example notebook.


In [ ]:
// Given: one import this section's checks rely on.
use std::sync::Arc;

/// Sum a Vec<i32> that arrived wrapped in an Arc — the shape a spawn
/// hands you when the data must be shared, not moved.
fn sum_arc(data: Arc<Vec<i32>>) -> i64 {
    todo!()
}

/// Multiply every element of `nums` by `factor`, using thread::scope to
/// fan the mutation out: split nums into chunks_mut, one spawn per chunk.
/// The scope's implicit joins make the borrows end before this returns.
fn scale_scoped(nums: &mut [i32], factor: i32) {
    todo!()
}

// --- check (do not modify) ---
let data = Arc::new(vec![1, 2, 3, 4]);
// the Arc crosses a thread boundary right here — that is the point
let inner = Arc::clone(&data); // the spawn's copy; `data` stays ours to inspect
let handle = thread::spawn(move || sum_arc(inner));
assert_eq!(handle.join().unwrap(), 10);
assert_eq!(Arc::strong_count(&data), 1); // the spawned copy is gone

let mut nums = vec![1, 2, 3, 4, 5, 6];
scale_scoped(&mut nums, 10);
assert_eq!(nums, vec![10, 20, 30, 40, 50, 60]);
println!("Exercise 2.1 passed");


## 3. Channels: Ownership as the Message Protocol

`send` moves; dropping the last `Sender` closes the channel; the error kinds are lifecycle signals, not failures.

**See**: §3 of the example notebook.


In [ ]:
// Given: mpsc arrives by path in the signatures; nothing new to import.

/// Square a fill quantity on a worker thread the CHANNEL way: move `qty`
/// through an mpsc channel, receive it in a spawned thread, send the square
/// BACK on a second channel, and return what you receive.
fn square_via_channel(qty: u64) -> u64 {
    todo!()
}

/// Fan N fills out to N worker threads (one Sender clone each), collect all
/// the squared quantities through the Receiver, and return them SORTED — the
/// set is guaranteed, the arrival order is not.
fn fan_out_squares(qtys: &[u64]) -> Vec<u64> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(square_via_channel(9), 81);
assert_eq!(fan_out_squares(&[3, 1, 2]), vec![1, 4, 9]);
assert_eq!(fan_out_squares(&[]), Vec::<u64>::new()); // zero senders -> instant close
println!("Exercise 3.1 passed");


In [ ]:
// Given: the error types come in with this section's one import.
use std::sync::mpsc::{sync_channel, RecvError, SendError};

/// Send `msg` to a channel whose Receiver has been dropped: report whether
/// send accepted it. (It must not — report the SendError as `false`.)
fn send_into_the_void(msg: u32) -> bool {
    todo!()
}

/// Receive from a channel whose senders are all gone and the queue is empty:
/// return the RecvError itself.
fn recv_after_close() -> RecvError {
    todo!()
}

/// A hand-to-hand transfer on a sync_channel(0): spawn a thread that sends
/// `qty`, receive it FIRST here (a capacity-0 send cannot complete alone),
/// then join the sender. Return the received value.
fn rendezvous(qty: u32) -> u32 {
    todo!()
}

// --- check (do not modify) ---
assert!(!send_into_the_void(404));
assert!(matches!(recv_after_close(), RecvError { .. }));
assert_eq!(rendezvous(7), 7);
assert!(matches!(
    {
        let (tx, rx) = std::sync::mpsc::channel::<u32>();
        drop(rx);
        tx.send(1)
    },
    Err(SendError(_))
));
println!("Exercise 3.2 passed");


## 4. `Mutex` and `RwLock`: Shared State Under Guard

The guard is the lock; RAII releases it; poisoning makes breakage explicit.

**See**: §4 of the example notebook.


In [ ]:
// Given: the guard types come in with the sync module import.
use std::sync::Mutex;

/// Add `qty` to the shared position behind the Mutex: lock, deref-mut, drop
/// the guard before returning (the RAII unlock).
fn add_to_position(position: &Mutex<i64>, qty: i64) {
    todo!()
}

/// Read the position without mutating: lock, deref, return the value.
fn read_position(position: &Mutex<i64>) -> i64 {
    todo!()
}

// --- check (do not modify) ---
let position = std::sync::Arc::new(Mutex::new(0i64));
let mut handles = Vec::new();
for k in 0..8 {
    let p = std::sync::Arc::clone(&position);
    handles.push(thread::spawn(move || {
        for _ in 0..100 {
            add_to_position(&p, k);
        }
    }));
}
for h in handles {
    h.join().unwrap();
}
assert_eq!(read_position(&position), 2800); // 8 threads x 100 x (0+..+7)
println!("Exercise 4.1 passed");


In [ ]:
// Given: poisoning needs only what §4.1 imported.

/// A crashed writer poisons the lock. Recover the data by hand: lock with
/// unwrap_or_else and into_inner, and return the value that was left behind.
fn recover_poisoned(poisoned: &Mutex<u32>) -> u32 {
    todo!()
}

// --- check (do not modify) ---
let ledger = std::sync::Arc::new(Mutex::new(100u32));
let ledger2 = std::sync::Arc::clone(&ledger);
let _ = thread::spawn(move || {
    let _g = ledger2.lock().unwrap();
    panic!("crashed mid-update, guard still held");
})
.join(); // the lock is poisoned now
assert_eq!(recover_poisoned(&ledger), 100);
println!("Exercise 4.2 passed");


In [ ]:
// Given: the reader/writer lock by path.

/// Take a write guard on `book`, push `qty`, and DROP the guard before
/// returning (a held write guard would deadlock the reader below).
fn append_quote(book: &std::sync::RwLock<Vec<i64>>, qty: i64) {
    todo!()
}

/// Take a read guard and return the LAST element (clone it out — the guard
/// borrows, the return value must not).
fn last_quote(book: &std::sync::RwLock<Vec<i64>>) -> i64 {
    todo!()
}

// --- check (do not modify) ---
let book = std::sync::RwLock::new(vec![100, 101]);
{
    let r1 = book.read().unwrap();
    let r2 = book.read().unwrap(); // two readers compose
    assert_eq!((r1[0], r2.len()), (100, 2));
}
append_quote(&book, 102);
assert_eq!(last_quote(&book), 102);
println!("Exercise 4.3 passed");


## 5. Atomics: Lock-Free Counters and Flags

`fetch_add` returns the previous value; `compare_exchange` is the CAS primitive whose `Err` hands you current reality.

**See**: §5 of the example notebook.


In [ ]:
// Given: the atomic types and orderings in one import.
use std::sync::atomic::{AtomicUsize, Ordering};

/// Atomically add `n` to `cell` and return the PREVIOUS value
/// (the read and the write are one operation). Relaxed is the right
/// ordering for a plain counter.
fn count_hits(cell: &AtomicUsize, n: u64) -> u64 {
    todo!()
}

/// Swap `cell` to zero and return what it held.
fn swap_zero(cell: &AtomicUsize) -> u64 {
    todo!()
}

/// The CAS loser's interface: write `new` only if the current value still
/// equals `expected`; return Ok(previous) on success, Err(current) on loss.
fn try_claim(cell: &AtomicUsize, expected: u64, new: u64) -> Result<u64, u64> {
    todo!()
}

// --- check (do not modify) ---
let hits = std::sync::Arc::new(AtomicUsize::new(0));
let mut handles = Vec::new();
for _ in 0..4 {
    let h = std::sync::Arc::clone(&hits);
    handles.push(thread::spawn(move || {
        for _ in 0..100 {
            count_hits(&h, 1);
        }
    }));
}
for h in handles {
    h.join().unwrap();
}
assert_eq!(hits.load(Ordering::Relaxed), 400);
let seq = AtomicUsize::new(10);
assert_eq!(count_hits(&seq, 5), 10);
assert_eq!(seq.load(Ordering::Relaxed), 15);
let cell = AtomicUsize::new(15);
assert_eq!(swap_zero(&cell), 15);
assert_eq!(cell.load(Ordering::Relaxed), 0);
let slot = AtomicUsize::new(100);
assert_eq!(try_claim(&slot, 100, 200), Ok(100));
assert_eq!(try_claim(&slot, 100, 999), Err(200)); // Err carries current reality
println!("Exercise 5.1 passed");


## 6. The Problem Threads Solve Poorly

*(Reading section — the example notebook's §6.1 timing cell is the demonstration, and this notebook never asserts a timing. The exercise work that matters from §6 is §7's: build the future, drive the future, feel the difference between a thread and a frame.)*

**See**: §6 of the example notebook.


## 7. `async`/`await`: Futures Are Frames

An `async fn` call constructs a state machine and runs nothing; a runtime polls it; the machine's size is exactly the locals that must survive its awaits.

**See**: §7 of the example notebook.


In [ ]:
// Given: the demo futures from the example notebook, reused here.
// (no imports — `thread` and `tokio` persist; Duration arrives with §9.2)

async fn add(a: i64, b: i64) -> i64 {
    a + b
}

async fn identity_unit() {}

async fn hold_string(s: String) -> usize {
    let n = s.len();
    identity_unit().await;
    n
}

/// Drive `add(a, b)` to completion on a fresh runtime (Runtime::new +
/// block_on — the manual form; the kernel owns main).
fn drive_add(a: i64, b: i64) -> i64 {
    todo!()
}

/// The frame made visible: construct hold_string's future over `s` and
/// return its size in bytes (size_of_val on the future).
fn frame_size_of_string_future(s: &str) -> usize {
    todo!()
}

/// Fan the quantities out as spawned tasks (one tokio::spawn per element,
/// squaring it), await every JoinHandle, and return the sum. Copy the slice
/// into owned data first — a spawn's future must be 'static.
fn total_of_squares(nums: &[u64]) -> u64 {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(drive_add(20, 22), 42);
assert_eq!(frame_size_of_string_future("abcdefghij"), 64);
assert_eq!(total_of_squares(&[1, 2, 3, 4]), 30);
println!("Exercise 7.1 passed");


In [ ]:
// Given: nothing new — identity_unit persists from §7.1.

/// The async stub is the exercise: a 4-KiB array local that lives ACROSS an
/// await, then returns its length. Await identity_unit() as the suspension.
async fn big_frame() -> usize {
    todo!()
}

/// Construct big_frame's future and return its size in bytes. Compare with
/// the 12-byte future of add: the frame is the locals it must keep alive.
fn size_of_big_frame() -> usize {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(size_of_big_frame(), 4098);
let rt = tokio::runtime::Runtime::new().unwrap();
assert_eq!(rt.block_on(big_frame()), 4096);
println!("Exercise 7.2 passed");


## 8. The Runtime: `tokio`, `spawn`, `JoinHandle`

A spawned task is a frame on a scheduler; its `JoinHandle` is a Future of a `Result` — panic or cancellation included in the type.

**See**: §8 of the example notebook.


In [ ]:
// 8.1 Given: tokio persists from §7.1's checks.

/// Square a quantity on the runtime: tokio::spawn a task that returns
/// `qty * qty`, await the JoinHandle, unwrap the Result, return it.
async fn spawn_square(qty: u64) -> u64 {
    todo!()
}

/// Fan the quantities out as independent TASKS (one spawn per element,
/// squaring it), await every handle, return the SUM. The set is certain;
/// the order is not — a sum does not care. Copy the slice into owned data
/// first: a spawn's future must be 'static.
async fn sum_spawned_squares(nums: &[u64]) -> u64 {
    todo!()
}

// --- check (do not modify) ---
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    assert_eq!(spawn_square(9).await, 81);
    let s = sum_spawned_squares(&[1, 2, 3, 4]).await;
    assert_eq!(s, 30);
});
println!("Exercise 8.1 passed");


In [ ]:
// 8.2 Given: block_in_place-style valves need the runtime the checks build.

/// The safety valve: run CPU-bound `total_sync` OFF the async workers.
/// tokio::spawn_blocking a closure that calls it, await the handle, unwrap,
/// return. (You are inside async code here — the handle await is available.)
async fn offload_sync_total(nums: Vec<u64>) -> u64 {
    todo!()
}

/// Total a slice with a plain synchronous loop — given CPU-bound work that
/// must go through spawn_blocking from async code (see 8.1's check).
fn total_sync(nums: &[u64]) -> u64 {
    todo!()
}

/// Cancellation has a type: tokio::spawn a 60-second sleep task, abort() it
/// at once, await the handle, and report whether it ended CANCELLED (true).
async fn abort_report() -> bool {
    todo!()
}

// --- check (do not modify) ---
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    assert_eq!(offload_sync_total(vec![1, 2, 3, 4]).await, 10);
    assert_eq!(total_sync(&[1, 2, 3, 4]), 10);
    assert!(abort_report().await);
});
println!("Exercise 8.2 passed");


## 9. `select!` and `timeout`: Racing Futures

Two futures, one winner; the losers are dropped at their next await. `timeout` is the deadline as a type.

**See**: §9 of the example notebook.


In [ ]:
// 9.1 Given: nothing new — a sync fn creates its OWN runtime (the manual
// form); call it from plain sync context, never inside another block_on.

/// Race two futures with tokio::select!: a sleep that wins in `fast_ms`
/// against one that wins in `slow_ms` (margin 10x). Return "fast" if the
/// fast arm won (it always must — pin the policy, not the timing).
/// You are a sync fn: build a Runtime, block_on the select!, drop the rt.
fn race_arms(fast_ms: u64, slow_ms: u64) -> &'static str {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(race_arms(5, 50), "fast");
println!("Exercise 9.1 passed");


In [ ]:
// 9.2 Given: this section's one import (the example notebook's §9 needs it too).
use std::time::Duration;

/// Wait `wait_ms` on a sleep under a `limit_ms` timeout: report whether the
/// future finished IN time (true) or was dropped by the deadline (false).
fn finishes_in_time(wait_ms: u64, limit_ms: u64) -> bool {
    todo!()
}

/// The reply-slot: tokio::spawn a task that sends `qty` on a oneshot
/// channel, await the receiver, unwrap, return the value.
async fn oneshot_roundtrip(qty: u64) -> u64 {
    todo!()
}

/// The bounded channel with backpressure: drain 1..=n sent on a
/// tokio::sync::mpsc::channel(4) by a spawned task, in order, returning them.
async fn drain_bounded(n: u64) -> Vec<u64> {
    todo!()
}

// --- check (do not modify) ---
// sync stubs first: each builds its own runtime, so call from SYNC context
assert!(finishes_in_time(5, 100));
assert!(!finishes_in_time(100, 5));
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    assert_eq!(oneshot_roundtrip(42).await, 42);
    assert_eq!(drain_bounded(3).await, vec![1, 2, 3]);
});
println!("Exercise 9.2 passed");


## 10. Blocking vs Async: The Rules of the Road

One decision per function: brief std-`Mutex` guards never cross an await; long sync sections go to `block_in_place`; shared state awaited-across wants `tokio::sync::Mutex`.

**See**: §10 of the example notebook.


In [ ]:
// 10.1 Given: the std guard by alias, as in the example notebook.
use std::sync::Mutex as StdMutex;

/// The BRIEF case: push `qty` onto the ledger behind a std Mutex. Lock,
/// push, and make sure the guard is gone before this returns — scope it.
fn record_fill(ledger: &StdMutex<Vec<i64>>, qty: i64) {
    todo!()
}

/// The LONG case: apply `factor` to every element synchronously, from async
/// context, WITHOUT stalling the worker: tokio::task::block_in_place the
/// whole section. (The multi-thread runtime the checks build is required.)
async fn scale_ledger(ledger: &StdMutex<Vec<i64>>, factor: i64) {
    todo!()
}

// --- check (do not modify) ---
let ledger = StdMutex::new(vec![10i64, 20]);
record_fill(&ledger, 30);
assert_eq!(ledger.lock().unwrap().as_slice(), &[10, 20, 30]);
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    scale_ledger(&ledger, 10).await;
});
assert_eq!(ledger.lock().unwrap().as_slice(), &[100, 200, 300]);
println!("Exercise 10.1 passed");


In [ ]:
// 10.2 Given: the tokio Mutex is reached by path.

/// The ACROSS-AWAIT case: add `qty` to the shared total guarded by a
/// tokio::sync::Mutex — `lock()` is itself an await here, because the guard
/// is not Send. Loop `times` times, adding each time.
async fn add_shared(shared: std::sync::Arc<tokio::sync::Mutex<u64>>, qty: u64, times: u32) {
    todo!()
}

// --- check (do not modify) ---
let shared = std::sync::Arc::new(tokio::sync::Mutex::new(0u64));
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let a = std::sync::Arc::clone(&shared);
    let b = std::sync::Arc::clone(&shared);
    tokio::join!(add_shared(a, 10, 2), add_shared(b, 100, 3));
    assert_eq!(*shared.lock().await, 320); // 2x10 + 3x100, interleaved safely
});
println!("Exercise 10.2 passed");


## 11. Mixed Review

Three problems, each from a different corner of the chapter. Everything needed is in the sections named.

**See**: §3 and §9 of the example notebook (11.1), §2 and §7 (11.2), §4 and §8 (11.3).


In [ ]:
// 11.1 Given: both channel worlds in one problem. Imports persist.

/// The pipeline: SYNC producers fan quantities in over a std mpsc channel
/// (one Sender clone per fill, moved across its own spawned thread); collect
/// them to completion (drop the last Sender — the iter ends), then run the
/// ASYNC stage: inside rt.block_on, one tokio::spawn per quantity squaring
/// it, await every JoinHandle, and return the squares SORTED.
fn pipeline_squares(qtys: &[u64]) -> Vec<u64> {
    todo!()
}

// --- check (do not modify) ---
assert_eq!(pipeline_squares(&[4, 1, 3]), vec![1, 9, 16]);
assert_eq!(pipeline_squares(&[]), Vec::<u64>::new());
println!("Exercise 11.1 passed");


In [ ]:
// 11.2 Given: scope borrows plus a future — the two "no copy" worlds.

/// Fan the MUTATION out with thread::scope (one spawn per chunk of 2) —
/// the scope's implicit joins end the borrows — then compute the sum INSIDE
/// rt.block_on(async { ... }) (borrows of nums are fine on one task).
/// Return the sum of the SCALED slice (each element x factor).
fn scale_then_sum(nums: &mut [i64], factor: i64) -> i64 {
    todo!()
}

// --- check (do not modify) ---
let mut nums = vec![1, 2, 3, 4];
assert_eq!(scale_then_sum(&mut nums, 10), 100);
assert_eq!(nums, vec![10, 20, 30, 40]); // the mutation really happened
println!("Exercise 11.2 passed");


In [ ]:
// 11.3 Given: the guarded state, the workers, and the deadline — the desk's day.

/// A worker's shift: on its OWN thread, add each order quantity to the
/// shared position behind its std Mutex. Spawn the thread and return its
/// JoinHandle — the check joins every shift before asserting the total.
fn run_shift(position: std::sync::Arc<StdMutex<i64>>, orders: Vec<i64>) -> std::thread::JoinHandle<()> {
    todo!()
}

// --- check (do not modify) ---
let position = std::sync::Arc::new(StdMutex::new(0i64));
let shifts: Vec<_> = (0..4)
    .map(|k| run_shift(std::sync::Arc::clone(&position), vec![k + 1; 10]))
    .collect();
for s in shifts {
    s.join().unwrap();
}
assert_eq!(*position.lock().unwrap(), 100); // 10x(1+2+3+4) across 4 joined shifts
println!("Exercise 11.3 passed");


## 12. What Comes Next

Every remaining applied-track chapter drives futures through a runtime.

| Next | What it borrows from here |
|---|---|
| **17 — HTTP Clients and WebSockets** | `reqwest`'s blocking vs async client split is §10 made manifest; the live price feed is `select!` + `timeout` + bounded channels. |
| **18 — Redis** | async connections on the §8 runtime; §9's bounded-channel backpressure becomes the pipeline discipline. |
| **19 — PostgreSQL** | pools are `Arc`-shared guarded state with `lock().await` semantics; transactions must survive `await` points. |
| **20 — PyO3** | the escape hatch back to Python — the GIL returns, and with it the one-thread-at-a-time world this chapter left. |

Carry forward: **the await point is the suspension point; the guard drop is the unlock; the dropped future is the cancellation.**
